Chat History -- Store the chat history for agent model think and answer the questions

In [8]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableWithMessageHistory, RunnableConfig, ConfigurableFieldSpec
from langchain_openai import ChatOpenAI
from langchain_community.chat_message_histories import ChatMessageHistory
from config.load_key import load_envkey

prompt = ChatPromptTemplate.from_messages([
    ("system","You're an assistant who's good at {ability}. Respnd in 20 words or fewer"),
    MessagesPlaceholder(variable_name="history"), # history message placeholder
    ("user", "{input}")
])
model = ChatOpenAI(
    model=load_envkey('MODEL_NAME'),
    base_url=load_envkey('BASE_URL'),
    api_key=load_envkey('API_KEY_DASHSCOPE')
)

Store the chat history in memory.
Use session id for identify

In [9]:
runnable = prompt | model

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(
    runnable,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

Example: set session id ""

In [14]:
# shared the same session id
response = with_message_history.invoke(
    {"ability": "math", "input": "余弦是什么意思？"},
    config={"configurable": {"session_id": "abc123"}}
)
print(response)

# not shared the same session id, no memory
response = with_message_history.invoke(
    {"ability": "math", "input": "什么？"},
    config={"configurable": {"session_id": "abc123"}}
)
print(response)

content='余弦（cos）是直角三角形中，一个锐角的邻边与斜边的比值。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 108, 'total_tokens': 133, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen-plus', 'system_fingerprint': None, 'id': 'chatcmpl-2b3f4af3-adc4-938a-adbb-c8cc8d42f8ea', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019bc0c3-0522-7df2-b3bb-364b84082948-0' usage_metadata={'input_tokens': 108, 'output_tokens': 25, 'total_tokens': 133, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}
content='余弦（cos）是直角三角形中，某个角的邻边除以斜边的值。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 145, 'total_tokens': 169, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen-plu

Example: set another session id "", it thought without chat history.

In [13]:
response = with_message_history.invoke(
    {"ability": "math", "input": "什么？"},
    config={"configurable": {"session_id": "def234"}}
)
print(response)

content='抱歉，我不太明白你的问题。能再解释清楚一点吗？' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 34, 'total_tokens': 49, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen-plus', 'system_fingerprint': None, 'id': 'chatcmpl-edf73e53-e8ad-9420-ac2f-56b2e9f71d2a', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019bc0c2-bee3-7280-b03b-2499622b7bd3-0' usage_metadata={'input_tokens': 34, 'output_tokens': 15, 'total_tokens': 49, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}
